# Lora no Kaggle: quatro tamanhos de modelo na GPU T4

Treina o extrator de pedidos com o [Soup CLI](https://github.com/MakazhanAlpamys/Soup) em quatro tamanhos
(Qwen2.5 de 0,5B, 1,5B, 3B e 7B) e avalia todos no mesmo conjunto de teste, separando casos normais e casos de borda.

Antes de rodar, no painel lateral **Settings**:

1. **Accelerator**: `GPU T4 x2`
2. **Internet**: ligada (precisa de telefone verificado na conta)

Para rodar sem deixar o navegador aberto, use **Save Version → Save & Run All (Commit)**.
O total estimado é de 4 horas de GPU, mas essa estimativa não foi medida: confira o tempo do primeiro modelo.

## 1. Conferir a GPU

In [ ]:
!nvidia-smi

## 2. Baixar o projeto e instalar o Soup

In [ ]:
# Pode rodar esta célula quantas vezes quiser: ela sempre volta para a pasta
# certa e atualiza o projeto, em vez de clonar de novo dentro dele.
%cd /kaggle/working
!rm -rf Lora/Lora
![ -d Lora/.git ] && git -C Lora pull -q || git clone -q https://github.com/FabioAGC/Lora.git
%cd /kaggle/working/Lora
!pip install -q "soup-cli[train]"
# O Kaggle traz um torchao antigo que impede o peft de carregar. O projeto não usa torchao.
!pip uninstall -y -q torchao


## 3. Treinar e avaliar

Usa uma só das duas T4. **Rode primeiro só o menor modelo** (cerca de 10 minutos): ele confirma que a GPU,
o Soup e a avaliação funcionam antes de você gastar horas. Depois que der certo, troque `MODELOS` por
`"0.5b 1.5b 3b 7b"` (ou só os que quiser) e rode de novo: modelos já avaliados são pulados.


In [ ]:
MODELOS = "0.5b"  # depois do primeiro teste: "0.5b 1.5b 3b 7b"
%env CUDA_VISIBLE_DEVICES=0
%env PYTHONPATH=src
!python -m pedido_lora.kaggle --modelos {MODELOS}


## 4. Resultado

In [ ]:
from IPython.display import Markdown
Markdown(open("results/tabela.md").read())

## 5. Testar uma mensagem com o maior modelo

In [ ]:
!python -m pedido_lora.inferir --modelo Qwen/Qwen2.5-7B-Instruct --quant4 --adaptador outputs/qwen_7b_qlora "me ve 2 xburguer sem cebola, nao pera, 3, e uma coquinha zero. quanto tempo demora a entrega?"

## 6. Guardar o resultado

Tudo que fica em `/kaggle/working` aparece na aba **Output** do notebook depois do commit, para baixar.
Os adaptadores LoRA são pequenos (dezenas de MB), então cabem com folga.

In [ ]:
!rm -rf /kaggle/working/resultado && mkdir -p /kaggle/working/resultado/adaptadores
!for d in outputs/qwen_*; do mkdir -p /kaggle/working/resultado/adaptadores/$(basename $d) && cp $d/adapter_* $d/*.json /kaggle/working/resultado/adaptadores/$(basename $d)/ 2>/dev/null; done
!cp -r results outputs/previsoes /kaggle/working/resultado/
!du -sh /kaggle/working/resultado/*

## 7. Publicar no Hugging Face (opcional)

1. Crie um token do tipo **Write** em huggingface.co/settings/tokens.
2. No Kaggle, em **Add-ons → Secrets**, crie um segredo com o Label `HF_TOKEN` e o token como valor, e **marque a caixinha** para anexá-lo a este notebook.
3. Rode a célula abaixo. Ela descobre seu usuário pelo token e publica o modelo escolhido em `MODELO`.

O model card (`docs/MODEL_CARD.md`) descreve a primeira versão: a célula troca o modelo base, mas a tabela de
resultados dele precisa ser atualizada à mão com os números da etapa 4.


In [ ]:
# Ajuste só estas duas linhas, se precisar.
NOME_DO_SEGREDO = "HF_TOKEN"   # o nome exato (Label) que você deu ao segredo no Kaggle
MODELO = "0.5b"                # qual modelo publicar: "0.5b", "1.5b", "3b" ou "7b"

import os, pathlib, subprocess, sys
%cd /kaggle/working/Lora

from kaggle_secrets import UserSecretsClient
try:
    token = UserSecretsClient().get_secret(NOME_DO_SEGREDO)
except Exception as erro:
    raise SystemExit(
        f"Não consegui ler o segredo '{NOME_DO_SEGREDO}' ({type(erro).__name__}). Confira em Add-ons > Secrets: "
        "1) o Label é exatamente esse, com maiúsculas iguais; "
        "2) a caixinha ao lado dele está marcada, para anexá-lo a ESTE notebook.")
os.environ["HF_TOKEN"] = token.strip()

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "huggingface_hub"], check=True)
from huggingface_hub import HfApi
try:
    usuario = HfApi().whoami()["name"]
except Exception as erro:
    raise SystemExit(f"O Hugging Face recusou o token ({erro}). Gere um token do tipo Write em "
                     "huggingface.co/settings/tokens e atualize o segredo.")

pastas = {"0.5b": ("outputs/qwen_0.5b", "Qwen/Qwen2.5-0.5B-Instruct"),
          "1.5b": ("outputs/qwen_1.5b", "Qwen/Qwen2.5-1.5B-Instruct"),
          "3b": ("outputs/qwen_3b", "Qwen/Qwen2.5-3B-Instruct"),
          "7b": ("outputs/qwen_7b_qlora", "Qwen/Qwen2.5-7B-Instruct")}
adaptador, base = pastas[MODELO]
if not pathlib.Path(adaptador, "adapter_model.safetensors").exists():
    treinados = [m for m, (pasta, _) in pastas.items() if pathlib.Path(pasta, "adapter_model.safetensors").exists()]
    raise SystemExit(f"O modelo {MODELO} ainda não foi treinado nesta sessão. Treinados: {treinados or 'nenhum'}.")

card = pathlib.Path("docs/MODEL_CARD.md")
card.write_text("\n".join(f"base_model: {base}" if l.startswith("base_model:") else l
                          for l in card.read_text().splitlines()) + "\n")
print(f"Publicando {adaptador} na conta {usuario}...")
subprocess.run([sys.executable, "-m", "pedido_lora.publicar", "--usuario", usuario, "--adaptador", adaptador],
               check=True, env={**os.environ, "PYTHONPATH": "src"})
